# Tutorial 160 — Create a custom QAChat tool

[Tutorial 150](./150_project_types.ipynb) introduced project types and intents.
Most intents ultimately answer through QAChat's ordinary retrieval and answering
pipeline. Some requests, however, need an explicit operation: inspect every
passage, produce a file with a fixed structure, count something exactly, or
apply deterministic validation.

QAChat handles these operations with **tools**. A tool is registered Python code
that an intent can select instead of ordinary retrieval and answer generation.
The relationship is:

![QAChat tool execution flow](160_tool_execution_flow.svg)

This tutorial first follows that relationship through QAChat's shipped
`generate_paper_summary` tool. The tool turns one paper into a structured report
using a Markdown template, validates the generated structure in Python, and
saves the result in the project's `outputs/` folder. You will run it first with
stand-in services and optionally with a real model.

You will then implement a minimal `word_count` tool yourself. This separates the
parts that every tool shares—the contract, settings validation, registration,
and intent configuration—from the more specialized logic of the research
briefing example.

As in [Tutorial 150](./150_project_types.ipynb), this tutorial works with
QAChat's extension modules directly because it is showing how QAChat itself can
be extended.


## 1. Prepare QAChat

Each tutorial is independently runnable. These cells locate the repository,
make the local package importable, and call `prepare_notebook()` to verify the
project `.venv` environment. If Jupyter asks you to select the printed `.venv`
kernel, select it and rerun the cell.

In [ ]:
from pathlib import Path

root = Path.cwd()
if not (root / "qachat").is_dir():
    root = root.parent

print(f"QAChat root: {root}")

In [ ]:
import sys

if str(root) not in sys.path:
    sys.path.insert(0, str(root))

In [ ]:
from setup.notebook import prepare_notebook

root = prepare_notebook(root)

In [ ]:
import qachat

## 2. See where a tool enters the request path

[Tutorial 150](./150_project_types.ipynb) showed that the router selects an
**intent**. Most intents then use the history and retrieval settings declared in
the project type. A tool-based intent takes a different path: its `tool` field
names a registered Python operation and supplies that operation's settings.

Open
[`qachat/project_types/literature.json`](../qachat/project_types/literature.json)
and inspect the `paper_summary` intent. It is selected only when the user asks
for a formatted summary saved as a file.


In [ ]:
import json
from qachat.project_type import PROJECT_TYPES_DIR

literature_project_type = json.loads(
    (PROJECT_TYPES_DIR / "literature.json").read_text(encoding="utf-8")
)

### 2.1 Inspect a tool-based intent

The `tool` object identifies `generate_paper_summary` and configures how that
operation runs. The project-type file does not import Python or name an
arbitrary module.

Inspect the complete intent before following the tool name into the registry.


In [ ]:
print(
    json.dumps(
        literature_project_type["intents"]["popular_science_summary"],
        indent=2,
    )
)

### 2.2 Follow the tool into the registry

The controlled boundary between project configuration and executable code is
[`qachat/tools/_registry.py`](../qachat/tools/_registry.py).

A project type can name only a tool present in `TOOL_REGISTRY`. Each registry
entry provides:

- the canonical tool name;
- a short description;
- the function that validates the tool's settings; and
- the function that runs the tool.

This means that JSON configuration can select and configure trusted operations,
but it cannot import arbitrary Python code.

Inspect the currently registered tools.


In [ ]:
from qachat.tools import TOOL_REGISTRY

for name, tool in TOOL_REGISTRY.items():
    print(f"{name:24} {tool.summary}")

## 3. Understand the tool contract

Every registered tool provides a runner with the same general interface, defined
in [`qachat/tools/_contract.py`](../qachat/tools/_contract.py):

```text
run(settings, request, context) -> ToolResult
```

The runner receives three arguments and returns a `ToolResult`:

| Name | Type | Role |
|---|---|---|
| `settings` | Tool-specific settings dataclass | The validated configuration for this particular tool. |
| `request` | `ToolRequest` | What the user is asking the tool to do. |
| `context` | `ToolContext` | The QAChat services and project information available to the tool. |
| return value | `ToolResult` | The answer, evidence, summary, and other information returned to QAChat. |

Here, `request` and `context` are parameter names: they contain instances of
the `ToolRequest` and `ToolContext` classes. The return value is an instance of
`ToolResult`.

`settings` is slightly different because each tool defines its own settings
dataclass. For example, `generate_paper_summary` defines `SummarySettings` and
therefore has the more specific signature:

```text
run(
    settings: SummarySettings,
    request: ToolRequest,
    context: ToolContext,
) -> ToolResult
```

An important design principle is that a tool does not import `RAGSession` or
reach directly into application state. Everything it may need is provided
through `ToolContext`. This makes the tool's dependencies explicit and allows
tests to replace models or other services with simple stand-ins.

Inspect the dataclass fields that define the request, context, and result.


In [ ]:
from dataclasses import fields
from qachat.tools import ToolContext, ToolRequest, ToolResult

`ToolRequest` contains what the user asked and, when relevant, the earlier turns
that the request refers to.


In [ ]:
[field.name for field in fields(ToolRequest)]

`ToolContext` contains the services QAChat makes available to the operation. A
tool can use only the capabilities it needs.


In [ ]:
[field.name for field in fields(ToolContext)]

`ToolResult` is the common return value. It separates the user-facing answer
from the evidence and machine-readable summary saved with the turn.


In [ ]:
[field.name for field in fields(ToolResult)]

## 4. Work with report templates

The `generate_paper_summary` tool is a useful reference tool because it
combines model calls with deterministic structure checking. The report format
itself is not hard-coded in Python: it is described by a Markdown **template**.

Every project can have a `templates/` folder next to `sources/` and `outputs/`.
When that folder is first created, QAChat copies in the shipped templates from
`qachat/tools/generate_paper_summary/templates/`. Users can then edit those copies or add new `.md`
templates without changing the tool code.


### 4.1 Inspect a shipped template

Open the shipped
[`research_briefing.md`](../qachat/tools/generate_paper_summary/templates/research_briefing.md)
template, then print it from Python.


In [ ]:
from qachat.tools.generate_paper_summary import SHIPPED_TEMPLATES_DIR

briefing_file = SHIPPED_TEMPLATES_DIR / "research_briefing.md"

print(briefing_file.read_text(encoding="utf-8"))

A report template has a small declarative grammar:

1. **Front matter** is optional metadata between two `---` lines. It can define
   `title`, `description`, `max_words`, and `reference`.
2. The **headline** is the `#` line. A suffix such as `(max 90 characters)`
   makes the headline model-generated and gives it a limit; without a limit,
   the line is a fixed title.
3. **Sections** are `##` headings in the order required by the report. The text
   below each heading tells the model what belongs in that section.
4. **Limits** can constrain words or items, for example `(60–150 words)`,
   `(max 100 words)`, `(2–3 items)`, or `(3 items)`.
5. **Labelled lines**, such as `**Citation:** {citation}`, create one-line
   fields.
6. **Placeholders** identify content supplied deterministically by Python.
   `{citation}` comes from the paper, while `{references}` lets the model choose
   entries that Python then copies from the paper's reference list.

`load_template()` parses and validates a template before the tool uses it.


In [ ]:
from qachat.tools.generate_paper_summary import load_template

template = load_template("research_briefing")

Inspect the headline and total-length constraints parsed from the template.


In [ ]:
print(
    f"{template.title}: headline "
    f"of at most {template.headline_characters} characters, "
    f"at most {template.max_total_words} words in total"
)

Then inspect each section, its limit, and the source of its content.


In [ ]:
for section in template.sections:
    limit = section.words or section.items or ""
    print(f"- {section.name} {limit} ({section.source})")

### 4.2 Create a small custom template

A project-specific template uses the same format. Define a shorter `Lab Summary`
with two prose sections and a reading list selected from the paper's own
references.


In [ ]:
lab_summary = """---
title: Lab Summary
max_words: 300
---

# Lab Summary

## Question (max 40 words)
What the paper asks, in one or two sentences.

## Findings (60–120 words)
What it found, with the key numbers.

## Read Next (2 items)
Two papers to read next, from {references}.
"""

Parse the text exactly as QAChat would parse a template file.


In [ ]:
from qachat.tools.generate_paper_summary import parse_template

mine = parse_template(lab_summary, "lab_summary")

Inspect the parsed sections. The first two are model-written prose; `Read Next`
uses reference entries supplied by Python.


In [ ]:
for section in mine.sections:
    print(
        f"- {section.name}: words {section.words}, items "
        f"{section.items}, from {section.source}"
    )

Template errors are reported when the template is parsed. The error identifies
the invalid rule so that the format can be corrected before a tool runs.


In [ ]:
try:
    parse_template("## Findings (sixty words)\nWhat it found.", "draft")
except ValueError as error:
    print(error)

### 4.3 Validate a generated report

A prompt that says *follow this format exactly* is not a guarantee.
`check_report()` validates the generated Markdown in Python. It checks that
required sections are present once, appear in the correct order, contain
content, respect their word or item limits, and use an allowed headline length.


First build deterministic test content that satisfies the shipped template.

Define a helper that creates content within each section's limit. This is test
data, not a real research briefing.


In [ ]:
def filler(section):
    """Content that fits one section's limit."""

    if section.items:
        return "- First paper.\n- Second paper."

    low, high = section.words
    return " ".join(["word"] * ((low + high) // 2))

Assemble the report using the template headings.


In [ ]:
parts = ["# Plants reveal stress before it shows"]
for section in template.sections:
    if section.inline:
        parts.append(f"**{section.name}:** Smith, J. et al. A title.")
    else:
        parts.append(f"## {section.name}\n{filler(section)}")

fitting = "\n\n".join(parts)

print(fitting)

Validate the assembled report. A valid report passes without requiring another
model call.


In [ ]:
from qachat.tools.generate_paper_summary import check_report

print("valid:", check_report(fitting, template).valid)

An incomplete or overlong report fails with specific problems. The real tool
uses these messages to request targeted repairs rather than regenerating the
whole report blindly.


In [ ]:
broken = "# " + "x" * 100 + "\n## The Reason\nToo short."

print(broken)

In [ ]:
for problem in check_report(broken, template).problems():
    print("-", problem)

## 5. Prepare a small project workspace

The report tool works with three project folders that have different roles:

| Folder | Role |
|---|---|
| `sources/` | The project's knowledge base. Sources are normally indexed and searched, but tools can also work with their passages directly. |
| `outputs/` | Generated files and user-provided working documents. They are not indexed. |
| `templates/` | Report-format definitions used by tools. They are not indexed. |

Keeping generated files out of `sources/` prevents a report written by QAChat
from later returning as evidence for another answer.

This tutorial creates a small project under `sandbox/`, copies one included
paper into `sources/`, and adds the `Lab Summary` template. To keep the example
focused on the tool contract, it does not build an embedding or retrieval
index. Instead, it reads and chunks the source directly and supplies those
passages through `ToolContext`.

Start with a clean tutorial workspace and create its `sources/` and `outputs/`
folders.


In [ ]:
import shutil
from qachat.tools._shared.outputs import output_folder

project_root = root / "sandbox" / "tutorial_160_custom_tool"
if project_root.exists():
    shutil.rmtree(project_root)

sources = project_root / "sources"
sources.mkdir(parents=True)
outputs = output_folder(project_root)

print("Sources:", sources)
print("Outputs:", outputs)


Choose the included example paper used by the literature tutorials.


In [ ]:
paper = (
    root / "example_repos" / "literature_repo" / "papers" / "patil25deep.pdf"
)

print(paper)

Copy the paper into `sources/`. It remains a project source even though this
tutorial deliberately does not build a retrieval index.


In [ ]:
shutil.copy2(paper, sources / paper.name)


Create the project's `templates/` folder and save the custom template there.
The folder is initialized with QAChat's shipped templates on first use.


In [ ]:
from qachat.workspace_resources import templates_folder

templates = templates_folder(project_root)

(templates / "lab_summary.md").write_text(lab_summary, encoding="utf-8")

Read and chunk the source directly. This performs the source-reading and
chunking steps needed by the tool, but it does not create embeddings or a
persistent retrieval index.

The resulting passages are the same kind of `Chunk` objects that an indexed
project would expose through `ToolContext.chunks`.


In [ ]:
from qachat.retrievals import estimate_tokens

source_report = qachat.scan_repository(
    sources,
    extensions=(".pdf",),
    max_file_bytes=paper.stat().st_size,
)
source_files = qachat.read_repository_files(source_report)
source_units = qachat.extract_source_units(source_files)
passages = tuple(
    qachat.chunk_source_units(
        source_units,
        count_tokens=estimate_tokens,
        overlap_tokens=0,
    )
)
source_sizes = {
    source.path: (sources / source.path).stat().st_size
    for source in source_files
}
estimated_tokens = sum(estimate_tokens(p.text) for p in passages)

print(
    f"{paper.name}: {len(passages)} passages, "
    f"about {estimated_tokens:,} tokens"
)


Inspect the resulting project structure. The paper is an input under
`sources/`; `outputs/` is still empty and will receive the report generated by
the tool. The project also contains the shipped templates and the custom
`lab_summary.md` file.


In [ ]:
print(f"{project_root.name}/")

for path in sorted(project_root.rglob("*")):
    relative = path.relative_to(project_root)
    indent = "    " * len(relative.parts)
    suffix = "/" if path.is_dir() else ""
    print(f"{indent}{relative.name}{suffix}")

## 6. Keep bibliographic content deterministic

A structured report may contain information that the model should not invent.
For `generate_paper_summary`, the paper citation and reference-list entries are
handled separately from generated prose.

The citation is extracted from the PDF metadata and first page by Python.


In [ ]:
from qachat.tools.generate_paper_summary import paper_citation

paper_file = sources / paper.name

print(paper_citation(paper_file, passages))


The reference list is also split into entries by Python. Inspect a few of the
entries that the tool can offer to the model.


In [ ]:
from qachat.tools.generate_paper_summary import reference_entries

entries = reference_entries(passages)
print(f"{len(entries)} reference entries:")
for entry in entries:
    print("-", entry)


When a template contains `{references}`, the model sees numbered reference
entries and returns only the numbers it wants. Python copies the selected
entries into the report verbatim; invalid numbers are ignored.

This separation illustrates a general tool-design principle: use the model for
judgement or generation where it is useful, and keep deterministic operations
in Python when correctness can be checked directly.


## 7. Run the tool with stand-in services

Before loading a real model, run the complete tool with deterministic stand-ins.
This is also how the tool can be tested: the operation receives the same
`ToolRequest` and `ToolContext`, but model calls return predefined text.


### 7.1 Define a stand-in model

`generate_paper_summary` takes notes, writes report sections, and selects
references through separate model calls. The small function below returns a
known response for each kind of call.

The reference-selection reply deliberately returns `3, 99`. Entry 3 is valid;
99 is outside the extracted reference list and is ignored by Python. The prose
responses are intentionally simple, so the final structural check may also
report length problems.


In [ ]:
def scripted_model(system, prompt, max_new_tokens):
    """Return deterministic replies for the tool's model calls."""

    if "take notes" in system:
        return "- The method is called DIVA [1]"

    if "choose references" in system:
        return "3, 99"

    return "Summarized from the notes [1]."

### 7.2 Build the tool context

Now supply the services required by the tool. This example needs the source
passages, model calls, progress reporting, and tracing, but it does not need an
embedding or retrieval index.

`make_context()` accepts the model callable so the same context construction can
be reused later with a real model. The optional `model_id` keeps the run summary
accurate in both cases.


In [ ]:
from qachat.trace_logging import NullTraceLogger

steps = []


def make_context(model, model_id="scripted"):
    """Create the minimal tool context used in this tutorial."""

    return ToolContext(
        project_root=project_root,
        chunks=passages,
        graph=qachat.CodeGraph(entities=(), relations=()),
        rank=lambda text: [],
        retrieval_count=20,
        unsearchable=lambda indexed: [],
        source_files=lambda: source_sizes,
        ask_router=model,
        ask_model=model,
        write=model,
        model_id=model_id,
        progress=lambda message, **details: steps.append(message),
        done=lambda message, **details: steps.append(message),
        confirm=None,
        trace=NullTraceLogger(),
        trace_turn_id=None,
    )


### 7.3 Configure the tool and request

`ToolUse` pairs a registered tool name with its settings object. When QAChat
loads a project type, those settings have already been validated by the tool's
`parse_settings()` function. Here we construct the known-valid settings
dataclass directly so that the example remains explicit.


In [ ]:
from qachat.tools import ToolUse, run_tool
from qachat.tools.generate_paper_summary import SummarySettings

use = ToolUse(
    "generate_paper_summary",
    SummarySettings(),
)


The request names both the source paper and the custom template. Being
explicit matters when a project contains several papers or templates.


In [ ]:
question = f"turn {paper.name} into a lab summary"
request = ToolRequest(question, question)


### 7.4 Run and inspect the result

Run the registered tool with the request and stand-in context. No real language
model is loaded.


In [ ]:
result = run_tool(use, request, make_context(scripted_model))

The context collected the progress messages emitted while the tool ran. Inspect
them to see the operation as a sequence of explicit steps.


In [ ]:
for step in steps:
    print("·", step)

The `ToolResult.summary` records machine-readable facts about the run. Confirm
which template was selected, where the report was saved, and whether the
stand-in responses satisfied the template constraints.


In [ ]:
if "saved" not in result.summary:
    raise RuntimeError(
        "The tool did not save a report. Its explanation: " + result.answer
    )

print("template:", result.summary["template"])
print("saved:", result.summary["saved"])
print("valid:", result.summary["valid"])
print("problems:", result.summary["problems"])

Open the generated Markdown file and inspect its contents. The link displayed by
the next cell points to the same saved file.


In [ ]:
from IPython.display import display, Markdown

saved = project_root / result.summary["saved"]
relative_saved = saved.relative_to(root)
link = f"[Open the generated report](../{relative_saved.as_posix()})"
display(Markdown(link))
print(saved.read_text(encoding="utf-8"))

## 8. Optional: run the same tool with a real model

The deterministic run is sufficient to understand and test the tool contract.
This optional section replaces only the model service; the registered tool,
request, context structure, templates, validation, and output handling remain
the same.

Leave `RUN_REAL_MODEL = False` when running the complete notebook quickly. Set
it to `True` to generate a real research briefing. The first run may download
the selected model, and the operation can take several minutes because the tool
reads the paper in parts and validates each generated section.


In [ ]:
RUN_REAL_MODEL = False
real_result = None
chat = None

if not RUN_REAL_MODEL:
    print("Skipping optional real-model testing.")

In [ ]:
REQUESTED_BACKEND = qachat.BACKEND_AUTO
SELECTED_BACKEND = qachat.choose_backend(REQUESTED_BACKEND)

print("Requested backend:", REQUESTED_BACKEND)
print("Selected backend:", SELECTED_BACKEND)

Choose the canonical QAChat chat model for the resolved backend.


In [ ]:
model_id = (
    qachat.DEFAULT_CHAT_MODEL_MLX
    if SELECTED_BACKEND == qachat.BACKEND_MLX
    else qachat.DEFAULT_CHAT_MODEL_TORCH
)
print("Model:", model_id)

Load the model only when real-model testing is enabled.


In [ ]:
if RUN_REAL_MODEL:
    chat = qachat.ChatSession(
        model_id,
        backend=SELECTED_BACKEND,
        thinking=False,
        temperature=0.0,
        allow_download=True,
    )

Adapt `ChatSession` to the callable interface expected by `ToolContext`.
`side_reply()` makes the model call without changing the conversation history.


In [ ]:
from qachat.answering import side_reply


def model(system, prompt, max_new_tokens):
    """Call the chat model without modifying its conversation history."""

    return side_reply(
        chat,
        system,
        prompt,
        max_new_tokens=max_new_tokens,
    )

In [ ]:
briefing = ToolUse(
    "generate_paper_summary", SummarySettings(repair_attempts=2)
)

Request the shipped `Research Briefing` template for the same paper.


In [ ]:
question = f"turn {paper.name} into a research briefing"
request = ToolRequest(question, question)


Run the tool with the real model service. Pass the actual model ID into the
context so it is recorded correctly in the tool summary.


In [ ]:
if RUN_REAL_MODEL:
    steps.clear()
    real_result = run_tool(
        briefing,
        request,
        make_context(model, model_id=model_id),
    )

Inspect the progress messages emitted during the real run.


In [ ]:
if RUN_REAL_MODEL:
    print("\n".join(f"· {step}" for step in steps))

In [ ]:
if real_result is not None:
    if "saved" not in real_result.summary:
        raise RuntimeError(
            "The tool did not save a report. Its explanation: "
            + real_result.answer
        )
    print("valid:", real_result.summary["valid"])
    print("problems:", real_result.summary["problems"])

If the tool saved a report, display the final Markdown in the notebook.


In [ ]:
if real_result is not None:
    saved = project_root / real_result.summary["saved"]
    display(Markdown(saved.read_text(encoding="utf-8")))

Release the model before returning to the model-free custom-tool example.


In [ ]:
if chat is not None:
    chat.close()

In the desktop app, the same execution path begins when the router selects the
`paper_summary` intent. The GUI can additionally provide interactive
confirmation and PDF writing through `ToolContext`.

The reference example has now shown the complete tool lifecycle. The remaining
step is to implement that contract yourself.


## 9. Create a minimal custom tool

A new tool needs:

1. a settings dataclass;
2. a `parse_settings(data, where)` function;
3. a `run(settings, request, context) -> ToolResult` function;
4. a registry entry; and
5. an intent that names the registered tool.

The example below implements a deliberately small `word_count` tool. It reads a
file from `outputs/`, counts whitespace-separated words, and reports the most
frequent ones. It needs no model, which keeps the tool contract visible.


### 9.1 Define the settings

The settings dataclass contains only configuration that belongs to the tool.
Here it controls how many frequent words to report.


Define the settings: how many frequent words to include in the result.

In [ ]:
from dataclasses import dataclass


@dataclass(frozen=True, slots=True)
class WordCountSettings:
    """Settings of the example word-count tool.

    Attributes
    ----------
    top_words: int
        Number of frequent words to include in the result.

    """

    top_words: int = 5

### 9.2 Validate configuration

Project-type JSON is external configuration, so a tool should reject unknown
fields and validate allowed ranges. QAChat provides small helpers for these
checks.


In [ ]:
from qachat.tools._contract import check_int, reject_unknown


def parse_settings(data: dict, where: str) -> WordCountSettings:
    """Validate settings from a project type.

    Parameters
    ----------
    data: dict
        Settings from the intent's ``tool`` object, without ``name``.
    where: str
        Configuration location used in validation errors.

    Returns
    -------
    WordCountSettings
        Validated settings.

    """

    reject_unknown(where, data, ("top_words",))
    return WordCountSettings(
        top_words=check_int(where, data, "top_words", 5, 1, 50)
    )

### 9.3 Implement the operation

The operation receives only its validated settings, the request, and the
context. The request supplies the file name; the context supplies the project
root. The function therefore does not depend on notebook globals.

This teaching example counts whitespace-separated words rather than linguistic
tokens.


In [ ]:
from collections import Counter
from qachat.tools._shared.outputs import load_output


def run(
    settings: WordCountSettings,
    request: ToolRequest,
    context: ToolContext,
) -> ToolResult:
    """Count words and list the most frequent ones.

    Parameters
    ----------
    settings: WordCountSettings
        Validated settings of the tool.
    request: ToolRequest
        Request containing the output file name.
    context: ToolContext
        QAChat services available to the tool.

    Returns
    -------
    ToolResult
        Word count and most frequent words.

    """

    document = load_output(context.project_root, request.text.strip())
    words = " ".join(p.text for p in document.passages).split()
    frequent = Counter(word.casefold() for word in words)
    top = frequent.most_common(settings.top_words)

    context.done(f"Counted {len(words)} words")

    answer = f"{document.path} has about {len(words):,} words."
    answer += "\nFrequent words: " + ", ".join(
        f"{word} ({count})" for word, count in top
    )
    return ToolResult(answer, (), {"name": "word_count"})

### 9.4 Test the tool directly

Validate one settings object before running the operation. In the real registry,
this parsing happens when QAChat loads the project type.


In [ ]:
settings = parse_settings({"top_words": 3}, "word_count")

print(settings)


Build a `ToolRequest` containing the name of the generated report in
`outputs/`. This gives the minimal custom tool a real working document while
keeping the original paper in `sources/`.


In [ ]:
output_name = Path(result.summary["saved"]).name
request = ToolRequest(output_name, output_name)


Run the operation with the same stand-in context used above. No model is needed
because this tool is entirely deterministic.


In [ ]:
print(run(settings, request, make_context(scripted_model)).answer)

## 10. Register the custom tool

The notebook defined `word_count` locally so that its contract could be tested
without modifying QAChat. To make the tool available to project types:

1. Save the implementation as `qachat/tools/word_count.py`.
2. Register it in
   [`qachat/tools/_registry.py`](../qachat/tools/_registry.py): import the module,
   add `"word_count"` to `ToolName`, define a `TOOL_WORD_COUNT` constant, add
   it to `TOOL_CHOICES`, and add a `Tool(...)` entry using the module's
   `parse_settings` and `run` functions.
3. Export any registry symbols that are intended to be public from the relevant
   `__all__` definitions.
4. Add an intent to the appropriate project type and name the registered tool.

For example, a literature intent could contain:

```json
"word_count": {
  "description": "Count the words of a file in the outputs folder.",
  "history": {"policy": "none"},
  "retrieval": {"mode": "none"},
  "tool": {"name": "word_count", "top_words": 3}
}
```

Registration makes the Python operation **available**; the project type decides
**when it is selected**. The JSON can choose only registered tools, so adding a
project type never imports arbitrary code.

The same pattern can support substantially richer operations: compare complete
papers, extract methods or statistics, screen documents against criteria, or
check a manuscript against a reporting guideline.


## Complete the tutorial series

**Previous:** [Tutorial 150 — Define project types and route intents](./150_project_types.ipynb)

Tutorials 150 and 160 complete the extension layer introduced after the
core RAG pipeline. A project type declares the available behaviors and
routes requests to intents; a registered tool supplies explicit Python
behavior when an intent needs more control than ordinary retrieval and
answer generation.
